# Üretim Kalite Puanı Polinom Regresyon Modeli

Bu çalışmada üretim veri seti ayrıntılı biçimde incelenmiş, tüm değişkenlerin dağılımları ve aralarındaki ilişkiler görselleştirilmiş, malzeme kaynaşma ölçütü ile kalite puanı arasındaki ilişki farklı derecelerdeki polinom regresyon modelleriyle test edilmiş, çapraz doğrulama ile en iyi model seçilmiş ve karşılaştırma amacıyla Rastgele Orman Regresyonu da eklenmiştir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from matplotlib.patches import FancyArrowPatch

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold, learning_curve
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')
sns.set_palette('husl')

In [ ]:
np.set_printoptions(precision=4, suppress=True)

print('Tüm kütüphaneler başarıyla yüklendi.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

In [ ]:
df = pd.read_csv('/content/manufacturing.csv')
df.head()

Temperature (°C): Sıcaklık (Santigrat derece cinsinden üretim veya ortam sıcaklığı).

Pressure (kPa): Basınç (Kilopaskal cinsinden makineye uygulanan fiziksel basınç).

Temperature x Pressure: Sıcaklık ile basınç değerlerinin birbiriyle çarpımı.

Material Fusion Metric: Malzeme erime/kaynaşma ölçütü (Sıcaklık ve basınçla hesaplanan bir formül).

Material Transformation Metric: Malzeme dönüşüm ölçütü (Malzemenin yapısal değişim seviyesi).

Quality Rating: Kalite derecesi (Üretilen ürünün son kalite puanı).

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

## Keşifsel Veri Analizi

Bu bölümde her bir değişkenin dağılımı, değişkenler arasındaki ilişki ve kalite puanı ile olan bağlantıları çok yönlü görselleştirmelerle incelenmiştir.

In [ ]:
tum_sutunlar = ['Temperature (°C)', 'Pressure (kPa)', 'Temperature x Pressure', 'Material Fusion Metric', 'Material Transformation Metric', 'Quality Rating']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, sutun in enumerate(tum_sutunlar):
    sns.histplot(df[sutun], bins=30, kde=True, ax=axes[i], color='#2980b9')
    axes[i].set_title(f'{sutun} Dağılımı')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, sutun in enumerate(tum_sutunlar):
    sns.boxplot(y=df[sutun], ax=axes[i], color='#27ae60')
    axes[i].set_title(f'{sutun} Kutu Grafiği')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
target = 'Quality Rating'
corr_with_target = corr[target].sort_values(ascending=False)
corr_with_target

In [ ]:
sns.pairplot(df[tum_sutunlar], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle('Tüm Değişkenler Arası İkili İlişkiler', y=1.01)
plt.show()

In [ ]:
df.head()

In [ ]:
plt.subplot(1, 2, 1)
sns.histplot(df[target], bins=10, kde=True)
plt.title(f'{target} Dağılımı')

plt.subplot(1, 2, 2)
sns.boxplot(y=df[target])
plt.title(f'{target} Kutu Grafiği')
plt.tight_layout()
plt.show()

In [ ]:
print('IQR Aykırı Değer Analizi:')
for col in ['Temperature (°C)', 'Pressure (kPa)', 'Temperature x Pressure', 'Material Fusion Metric', 'Material Transformation Metric', 'Quality Rating']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: IQR={IQR:.2f}, Aralık=[{lower:.2f}, {upper:.2f}], Aykırı Değer Sayısı={len(outliers)}, Oran=%{len(outliers) / len(df) * 100:.2f}')

In [ ]:
df.head()

## Model Kurulumu

Korelasyon analizine göre kalite puanı ile en güçlü ilişkiye sahip değişken olan malzeme kaynaşma ölçütü kullanılarak farklı derecelerde polinom regresyon modelleri eğitilmiştir.

In [ ]:
X = df[['Material Fusion Metric']].values
y = df['Quality Rating'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

In [ ]:
print(f'X_train boyutu: {X_train.shape}')
print(f'y_train boyutu: {y_train.shape}')

In [ ]:
degrees = [1, 2, 3, 4, 5, 7, 10]
models = {}
results = []

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=True),
        LinearRegression()
    )

    model.fit(X_train, y_train)
    models[degree] = model

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    results.append({
        'Derece': degree,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
X_line = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)

In [ ]:
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

palette = sns.color_palette('husl', len(degrees))

num_samples = min(len(X_test), 1500)
sample_idx = np.random.choice(len(X_test), num_samples, replace=False)

for i, degree in enumerate(degrees):
    model = models[degree]
    y_line = model.predict(X_line)

    axes[i].scatter(X_train, y_train, alpha=0.3, s=15, color='gray', label='Eğitim Verisi')
    axes[i].scatter(X_test[sample_idx], y_test[sample_idx], alpha=0.4, s=15, color='orange', label='Test Verisi')
    axes[i].plot(X_line, y_line, color=palette[i], linewidth=2.5, label=f'{degree}. Derece')
    axes[i].set_title(f'{degree}. Derece Polinom')
    axes[i].set_xlabel('Material Fusion Metric')
    axes[i].set_ylabel('Quality Rating')
    axes[i].legend(fontsize=8)

axes[7].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Performans Karşılaştırması', fontsize=13, fontweight='bold')

x_ticks = results_df['Derece'].tolist()

axes[0].plot(results_df['Derece'], results_df['Train R²'], 'o-', color='#2ecc71', linewidth=2.5, markersize=7, label='Train R²')
axes[0].plot(results_df['Derece'], results_df['Test R²'], 'o-', color='#e74c3c', linewidth=2.5, markersize=7, label='Test R²')
axes[0].set_xlabel('Polinom Derecesi')
axes[0].set_ylabel('R² Skoru')
axes[0].set_title('R² Karşılaştırması')
axes[0].set_xticks(x_ticks)
axes[0].legend()

axes[1].plot(results_df['Derece'], results_df['Train RMSE'], 'o-', color='#2ecc71', linewidth=2.5, markersize=7, label='Train RMSE')
axes[1].plot(results_df['Derece'], results_df['Test RMSE'], 'o-', color='#e74c3c', linewidth=2.5, markersize=7, label='Test RMSE')
axes[1].set_xlabel('Polinom Derecesi')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Karşılaştırması')
axes[1].set_xticks(x_ticks)
axes[1].legend()

overfit_gap = results_df['Train R²'] - results_df['Test R²']
axes[2].bar(results_df['Derece'], overfit_gap, color='#9b59b6')
axes[2].set_xlabel('Polinom Derecesi')
axes[2].set_ylabel('Train R² - Test R²')
axes[2].set_title('Aşırı Öğrenme (Overfitting) Farkı')
axes[2].set_xticks(x_ticks)

plt.tight_layout()
plt.show()

## Çapraz Doğrulama

In [ ]:
print('5-Fold Cross-Validation Sonuçları:')
print('-' * 55)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree),
        LinearRegression()
    )
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

    cv_results.append({
        'Derece': degree,
        'Ortalama R²': cv_scores.mean(),
        'Standart Sapma': cv_scores.std()
    })
    print(f'{degree}. Derece: Ortalama R² = {cv_scores.mean():.4f}, Standart Sapma = {cv_scores.std():.4f}')

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(cv_results_df['Derece'], cv_results_df['Ortalama R²'], yerr=cv_results_df['Standart Sapma'], fmt='o-', color='#2980b9', capsize=5, linewidth=2)
plt.xlabel('Polinom Derecesi')
plt.ylabel('Ortalama R² (± Standart Sapma)')
plt.title('Çapraz Doğrulama Sonuçlarının Derece Bazında Değişimi')
plt.tight_layout()
plt.show()

In [ ]:
best_degree = int(results_df.loc[results_df['Test R²'].idxmax(), 'Derece'])
best_model = models[best_degree]

y_test_pred = best_model.predict(X_test)
y_train_pred = best_model.predict(X_train)
residuals = y_test - y_test_pred

print(f'Seçilen Model: {best_degree}. Derece Polinom')
print(f'Train R² = {r2_score(y_train, y_train_pred):.5f}')
print(f'Test R² = {r2_score(y_test, y_test_pred):.5f}')
print(f'Test RMSE = {np.sqrt(mean_squared_error(y_test, y_test_pred)):.5f}')

## Artık Analizi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test_pred, residuals, alpha=0.5, color='#8e44ad')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Tahmin Edilen Kalite Puanı')
axes[0].set_ylabel('Artıklar (Residuals)')
axes[0].set_title(f'{best_degree}. Derece Polinom Modeli Artık Grafiği')

sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color='#d35400')
axes[1].set_title('Artıkların Dağılımı')

stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Normal Olasılık Grafiği (QQ-Plot)')

plt.tight_layout()
plt.show()

## Öğrenme Eğrisi

Seçilen en iyi polinom modelinin eğitim veri miktarına göre performansının nasıl değiştiği incelenmiştir.

In [ ]:
egitim_boyutlari, egitim_skorlari, test_skorlari = learning_curve(
    make_pipeline(PolynomialFeatures(degree=best_degree), LinearRegression()),
    X, y, cv=kf, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 8), random_state=42
)

plt.figure(figsize=(10, 6))
plt.plot(egitim_boyutlari, egitim_skorlari.mean(axis=1), 'o-', color='#2ecc71', label='Eğitim Skoru')
plt.plot(egitim_boyutlari, test_skorlari.mean(axis=1), 'o-', color='#e74c3c', label='Doğrulama Skoru')
plt.xlabel('Eğitim Örnek Sayısı')
plt.ylabel('R² Skoru')
plt.title(f'{best_degree}. Derece Polinom Model Öğrenme Eğrisi')
plt.legend()
plt.tight_layout()
plt.show()

## Karşılaştırma İçin Rastgele Orman Regresyonu

Polinom regresyon modeliyle karşılaştırma yapmak amacıyla, doğrusal olmayan ilişkileri farklı bir mantıkla öğrenen Rastgele Orman Regresyonu da aynı veri üzerinde eğitilmiştir.

In [ ]:
orman_model = RandomForestRegressor(n_estimators=300, random_state=42)
orman_model.fit(X_train, y_train)
orman_tahmin = orman_model.predict(X_test)

orman_r2 = r2_score(y_test, orman_tahmin)
orman_rmse = np.sqrt(mean_squared_error(y_test, orman_tahmin))

son_karsilastirma = pd.DataFrame({
    'Model': [f'{best_degree}. Derece Polinom', 'Rastgele Orman'],
    'Test R²': [r2_score(y_test, y_test_pred), orman_r2],
    'Test RMSE': [np.sqrt(mean_squared_error(y_test, y_test_pred)), orman_rmse]
})
son_karsilastirma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(son_karsilastirma['Model'], son_karsilastirma['Test R²'], color='#16a085')
axes[0].set_title('Test R² Karşılaştırması')

axes[1].bar(son_karsilastirma['Model'], son_karsilastirma['Test RMSE'], color='#c0392b')
axes[1].set_title('Test RMSE Karşılaştırması')

plt.tight_layout()
plt.show()

## Sonuç

Çapraz doğrulama ve test performansı karşılaştırması sonucunda, düşük dereceli polinom modelleri aşırı öğrenmeye girmeden en dengeli performansı sağlamıştır. Yüksek dereceli modellerde eğitim performansı artsa da test performansının düşmesi aşırı öğrenmenin bir göstergesidir. Rastgele Orman Regresyonu ile yapılan karşılaştırma, tek değişkenli bu ilişki için polinom regresyonunun rekabetçi bir performans sunduğunu ortaya koymaktadır.